### Librerías utilizadas

| Librería | Para qué la usamos |
|---|---|
| `numpy` | Cálculo numérico y manejo de arreglos. |
| `pandas` | Cargar y manipular la tabla de datos (el `DataFrame`). |
| `matplotlib` | Generar los gráficos (*boxplots*). |
| `sklearn.preprocessing` | Las herramientas de codificación (`LabelEncoder`, `OneHotEncoder`) y escalamiento (`StandardScaler`). |

`np.random.seed(42)` fija la semilla aleatoria: garantiza que cualquier proceso con azar
dé **siempre el mismo resultado**, lo que asegura la *reproducibilidad* exigida en la fase.

In [1]:
import numpy as np                  # cálculo numérico y operaciones vectorizadas
import pandas as pd                 # estructuras tabulares: Series y DataFrame
import matplotlib.pyplot as plt     # gráficos de control
from sklearn import preprocessing    # LabelEncoder y OneHotEncoder
from sklearn.preprocessing import StandardScaler   # estandarización z-score

# Reproducibilidad: fijamos la semilla aleatoria del entorno
np.random.seed(42)

%matplotlib inline

## Configuración del entorno y de las rutas

Antes de cargar nada conviene comprobar el entorno y dejar declaradas las rutas.
Se usan rutas **relativas** a la raíz del repositorio, coherentes con la
estructura creada en la Fase 1: una ruta absoluta como `C:/Users/...` funciona en
un solo computador del mundo.

In [2]:
from pathlib import Path      # manejo de rutas independiente del sistema operativo
import sys

# sys.version trae la versión completa con fecha de compilación;
# split()[0] deja solo el número, que es lo único que hay que comprobar.
print("Python:", sys.version.split()[0])
for lib, mod in [("numpy", np), ("pandas", pd)]:
    print(f"{lib:8}:", mod.__version__)

# Estructura del proyecto. parents=True crea las carpetas intermedias;
# exist_ok=True evita el error si ya existen.
DIR_CRUDO = Path("Data/Raw")            # datos originales: nunca se modifican
DIR_PROCESADO = Path("Data/Processed")  # resultado del pipeline
DIR_DOCS = Path("docs")                 # diccionario, bitácora y metadatos
for carpeta in (DIR_CRUDO, DIR_PROCESADO, DIR_DOCS):
    carpeta.mkdir(parents=True, exist_ok=True)

ARCHIVO = DIR_CRUDO / "listings.csv.gz"
print("\nArchivo esperado en:", ARCHIVO)

Python: 3.13.5
numpy   : 2.5.3
pandas  : 3.0.5

Archivo esperado en: Data\Raw\listings.csv.gz


### Procedencia del conjunto de datos

| Campo | Valor |
| --- | --- |
| Título | Datos Airbnb Santiago Junio 2026 |
| Autor | Inside Airbnb |
| Plataforma | Inside Airbnb |
| Enlace | `https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/data/reviews.csv.gz` |
| Estructura esperada | 18300 filas x 90 columnas |
| Unidad de observación | Precio por noche |


In [3]:
# Reemplaza el nombre por el de tu archivo real
# Buscar el dataset desde la carpeta superior del proyecto
base = Path.cwd().parent

archivo = next(
    base.rglob("listings.csv.gz"),
    None
)

if archivo is None:
    raise FileNotFoundError(
        f"No se encontró el dataset dentro de:\n{base}"
    )

print("Dataset encontrado en:")
print(archivo)

df = pd.read_csv(archivo)

print(f"\nDataset cargado correctamente")
print(f"Dimensiones: {df.shape}")

df.head()

Dataset encontrado en:
C:\Users\felip\Documents\GitHub\proyecto-grupo3-mcdi500\Data\Raw\listings.csv.gz

Dataset cargado correctamente
Dimensiones: (18534, 90)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,978070332077815549,https://www.airbnb.com/rooms/978070332077815549,20260629151853,2026-07-01,city scrape,luminosa mansarda con balcón,Forget your worries in this spacious and seren...,NaN,https://a0.muscache.com/pictures/d0dda925-845a...,118157228,...,5.00,5.00,5.00,NaN,NaN,5,0,5,0,0.06
1,1069858035768058539,https://www.airbnb.com/rooms/1069858035768058539,20260629151853,2026-06-30,city scrape,Cómoda habitación bien ubicada con baño privado,From this centrally located place you can enjo...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,462786436,...,5.00,4.88,5.00,NaN,NaN,1,0,1,0,1.06
2,37181369,https://www.airbnb.com/rooms/37181369,20260629151853,2026-06-30,city scrape,Cómodo departamento Cerca de Metro,Comfortable apartment in a strategic location ...,NaN,https://a0.muscache.com/pictures/85358e52-06d7...,279796881,...,4.83,4.81,4.79,NaN,NaN,1,1,0,0,1.54
3,1257358306712011993,https://www.airbnb.com/rooms/1257358306712011993,20260629151853,2026-07-01,city scrape,Acogedor Dormitorio + baño priv.,Enjoy the simplicity of this quiet and central...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,149097696,...,NaN,NaN,NaN,NaN,NaN,1,0,1,0,NaN
4,868747298775934782,https://www.airbnb.com/rooms/868747298775934782,20260629151853,2026-07-01,city scrape,Departamento frente a est. metro,Enjoy a stylish experience in this centrally l...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,386871617,...,NaN,NaN,NaN,NaN,NaN,1,1,0,0,NaN


## 1. Obtención de los datos

**Qué hace este paso.** Lee el archivo CSV y lo carga en un `DataFrame` (la tabla con la
que trabajaremos).

**Por qué con una función.** Encapsulamos la lectura en `cargar_datos(ruta)` con un bloque
`try / except`. Así, si el archivo no existe, en lugar de un error técnico confuso, el
usuario recibe un mensaje claro indicando qué revisar. La función, además, imprime las
dimensiones cargadas, lo que sirve como primera verificación de que el archivo se leyó bien.

In [4]:
from pathlib import Path
import pandas as pd

# El notebook está en F2 y el dataset está en F1/data/raw
ARCHIVO = Path("../Data/Raw/listings.csv.gz")

print("Directorio actual:")
print(Path.cwd())

print("\nRuta del archivo:")
print(ARCHIVO.resolve())

print("\n¿Existe el archivo?")
print(ARCHIVO.exists())

# Validación
if not ARCHIVO.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo en:\n{ARCHIVO.resolve()}"
    )

# Cargar dataset
df_crudo = pd.read_csv(
    ARCHIVO,
    na_values=["N/A"]
)

# Mantener copia original
df = df_crudo.copy()

print("\nDatos cargados correctamente")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

df.head()

Directorio actual:
C:\Users\felip\Documents\GitHub\proyecto-grupo3-mcdi500\F2

Ruta del archivo:
C:\Users\felip\Documents\GitHub\proyecto-grupo3-mcdi500\Data\Raw\listings.csv.gz

¿Existe el archivo?
True

Datos cargados correctamente
Filas: 18534
Columnas: 90


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,978070332077815549,https://www.airbnb.com/rooms/978070332077815549,20260629151853,2026-07-01,city scrape,luminosa mansarda con balcón,Forget your worries in this spacious and seren...,NaN,https://a0.muscache.com/pictures/d0dda925-845a...,118157228,...,5.00,5.00,5.00,NaN,NaN,5,0,5,0,0.06
1,1069858035768058539,https://www.airbnb.com/rooms/1069858035768058539,20260629151853,2026-06-30,city scrape,Cómoda habitación bien ubicada con baño privado,From this centrally located place you can enjo...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,462786436,...,5.00,4.88,5.00,NaN,NaN,1,0,1,0,1.06
2,37181369,https://www.airbnb.com/rooms/37181369,20260629151853,2026-06-30,city scrape,Cómodo departamento Cerca de Metro,Comfortable apartment in a strategic location ...,NaN,https://a0.muscache.com/pictures/85358e52-06d7...,279796881,...,4.83,4.81,4.79,NaN,NaN,1,1,0,0,1.54
3,1257358306712011993,https://www.airbnb.com/rooms/1257358306712011993,20260629151853,2026-07-01,city scrape,Acogedor Dormitorio + baño priv.,Enjoy the simplicity of this quiet and central...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,149097696,...,NaN,NaN,NaN,NaN,NaN,1,0,1,0,NaN
4,868747298775934782,https://www.airbnb.com/rooms/868747298775934782,20260629151853,2026-07-01,city scrape,Departamento frente a est. metro,Enjoy a stylish experience in this centrally l...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,386871617,...,NaN,NaN,NaN,NaN,NaN,1,1,0,0,NaN


Mostramos las primeras filas para confirmar visualmente que las columnas se cargaron correctamente:

`head()` muestra las primeras cinco filas. Es la comprobación más barata que existe: confirma que el separador se interpretó bien, que los nombres de columna son los esperados y que los valores no quedaron corridos de columna.



## 2. Exploración inicial

**Qué hace este paso.** Antes de modificar nada, miramos el estado original de los datos.
Esto nos da la "línea base" contra la cual verificaremos después cada transformación.

La función `explorar_dataframe` reporta cuatro cosas:
1. **Dimensiones** (`shape`): cuántas filas y columnas hay.
2. **Tipos de datos** (`dtypes`): qué columnas son numéricas y cuáles son texto. Las de
   texto (`object`) son las que tendremos que codificar más adelante.
3. **Valores nulos** (`isnull().sum()`): cuántos datos faltan en cada columna. Aquí
   detectaremos que `bmi` tiene huecos.
4. **Estadísticos descriptivos** (`describe()`): media, mínimo, máximo, etc., de las
   variables numéricas, útil para detectar rangos raros o valores atípicos.

In [5]:
def explorar_dataframe(df):
    """Resumen exploratorio: dimensiones, tipos, nulos y estadisticos descriptivos."""
    print("Dimensiones:", df.shape)
    print("\nTipos de datos:")
    print(df.dtypes)
    print("\nValores nulos por columna:")
    print(df.isnull().sum())
    print("\nEstadisticos descriptivos (variables numericas):")
    return df.describe()


explorar_dataframe(df)

Dimensiones: (18534, 90)

Tipos de datos:
id                                                int64
listing_url                                         str
scrape_id                                         int64
last_scraped                                        str
source                                              str
                                                 ...   
calculated_host_listings_count                    int64
calculated_host_listings_count_entire_homes       int64
calculated_host_listings_count_private_rooms      int64
calculated_host_listings_count_shared_rooms       int64
reviews_per_month                               float64
Length: 90, dtype: object

Valores nulos por columna:
id                                                 0
listing_url                                        0
scrape_id                                          0
last_scraped                                       0
source                                             0
                       

,id,scrape_id,neighborhood_overview,host_id,host_profile_id,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
count,1.853400e+04,1.853400e+04,0.0,1.853400e+04,1.853400e+04,0.0,18534.000000,18534.000000,18534.000000,18534.000000,...,15243.000000,15243.000000,15242.000000,15243.000000,0.0,18534.000000,18534.000000,18534.000000,18534.000000,15243.000000
mean,1.092569e+18,2.026063e+13,NaN,2.041117e+16,1.485263e+18,NaN,6.041275,5.533614,3.908978,5.313100,...,4.859185,4.850596,4.829153,4.758378,NaN,12.727636,11.962609,0.717384,0.020287,1.983135
std,5.815742e+17,0.000000e+00,NaN,1.845515e+17,5.150438e+16,NaN,3.810797,3.317337,3.597149,3.395606,...,0.331657,0.339451,0.323088,0.388600,NaN,34.118329,34.188736,2.560209,0.255345,2.124072
min,4.939200e+04,2.026063e+13,NaN,1.961100e+04,1.462507e+18,NaN,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,1.000000,1.000000,NaN,1.000000,0.000000,0.000000,0.000000,0.010000
25%,8.277604e+17,2.026063e+13,NaN,1.042941e+08,1.465436e+18,NaN,2.000000,3.000000,1.000000,2.000000,...,4.850000,4.830000,4.790000,4.700000,NaN,1.000000,1.000000,0.000000,0.000000,0.490000
50%,1.310270e+18,2.026063e+13,NaN,2.756310e+08,1.469071e+18,NaN,6.000000,5.000000,3.000000,5.000000,...,4.940000,4.950000,4.920000,4.850000,NaN,2.000000,2.000000,0.000000,0.000000,1.340000
75%,1.551546e+18,2.026063e+13,NaN,5.241009e+08,1.470476e+18,NaN,9.000000,8.000000,7.000000,8.000000,...,5.000000,5.000000,5.000000,4.970000,NaN,8.000000,7.000000,0.000000,0.000000,2.810000
max,1.718110e+18,2.026063e+13,NaN,1.717781e+18,1.718110e+18,NaN,17.000000,11.000000,15.000000,11.000000,...,5.000000,5.000000,5.000000,5.000000,NaN,253.000000,253.000000,30.000000,7.000000,54.120000


También revisamos **cuántas categorías distintas** tiene cada variable de texto y cuántas
veces aparece cada una. Esto cumple dos funciones: detectar errores de tipeo o categorías
inesperadas, y conocer el orden en que aparecerán las categorías al codificarlas.

In [6]:
#se extrae los nombres de todas las variables, para determinar cuales serán objeto de estudio
#y cuales serán descartadas, quedando un respaldo en la data/raw, pero no pasando a las siguientes fases.

print(df.columns)

Index(['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
       'description', 'neighborhood_overview', 'picture_url', 'host_id',
       'host_url', 'host_profile_id', 'host_profile_url', 'host_name',
       'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months',
       'hosts_time_as_host_years', 'hosts_time_as_host_months',
       'host_location', 'host_about', 'host_response_time',
       'host_response_rate', 'host_acceptance_rate', 'host_is_superhost',
       'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood',
       'host_listings_count', 'host_total_listings_count',
       'host_verifications', 'host_has_profile_pic', 'host_identity_verified',
       'neighbourhood', 'neighbourhood_cleansed',
       'neighbourhood_group_cleansed', 'latitude', 'longitude',
       'property_type', 'room_type', 'accommodates', 'bathrooms',
       'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price',
       'price_quote_checkin_date', 'price_quote_c

In [7]:
#estas serán las columnas objeto de estudio
columnas_de_interes = ["hosts_time_as_host_years","neighbourhood_cleansed","property_type","room_type","accommodates", "bathrooms",
                      "bedrooms", "beds", 'price_quote_price_per_night','number_of_reviews', 'review_scores_rating', 'review_scores_accuracy', 
                       'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value' ]

In [8]:
df = df[columnas_de_interes]

In [9]:
explorar_dataframe(df)

Dimensiones: (18534, 17)

Tipos de datos:
hosts_time_as_host_years         int64
neighbourhood_cleansed             str
property_type                      str
room_type                          str
accommodates                     int64
bathrooms                      float64
bedrooms                       float64
beds                           float64
price_quote_price_per_night    float64
number_of_reviews                int64
review_scores_rating           float64
review_scores_accuracy         float64
review_scores_cleanliness      float64
review_scores_checkin          float64
review_scores_communication    float64
review_scores_location         float64
review_scores_value            float64
dtype: object

Valores nulos por columna:
hosts_time_as_host_years          0
neighbourhood_cleansed            0
property_type                     0
room_type                         0
accommodates                      0
bathrooms                      1747
bedrooms                       2337
b

,hosts_time_as_host_years,accommodates,bathrooms,bedrooms,beds,price_quote_price_per_night,number_of_reviews,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value
count,18534.000000,18534.000000,16787.000000,16197.000000,17510.000000,1.768700e+04,18534.000000,15243.000000,15243.000000,15243.000000,15243.000000,15243.000000,15242.000000,15243.000000
mean,3.908978,3.110446,1.307202,1.493239,2.098001,1.181977e+05,37.234920,4.784768,4.814539,4.753310,4.859185,4.850596,4.829153,4.758378
std,3.597149,1.817214,0.742421,1.026823,1.855753,1.226124e+06,66.592558,0.376882,0.367537,0.391464,0.331657,0.339451,0.323088,0.388600
min,0.000000,1.000000,0.500000,0.000000,1.000000,9.786400e+02,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,1.000000,2.000000,1.000000,1.000000,1.000000,4.150000e+04,2.000000,4.740000,4.790000,4.680000,4.850000,4.830000,4.790000,4.700000
50%,3.000000,3.000000,1.000000,1.000000,2.000000,5.900000e+04,13.000000,4.880000,4.910000,4.860000,4.940000,4.950000,4.920000,4.850000
75%,7.000000,4.000000,1.500000,2.000000,3.000000,9.078100e+04,44.000000,5.000000,5.000000,4.990000,5.000000,5.000000,5.000000,4.970000
max,15.000000,16.000000,22.000000,50.000000,57.000000,9.700004e+07,1388.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000


### El diccionario de variables

Antes de limpiar hay que declarar **qué es cada variable**. Esta es la decisión
que determina todo el preprocesamiento posterior.


| Rol analítico | Qué preprocesamiento exige |
| --- | --- |
| **Continua** | Detección de atípicos, imputación, escalamiento |
| **Binaria** | Codificación 0/1 y revisión de desbalance |
| **Nominal** | *One-hot encoding* |
| **Ordinal** | Codificación con el orden declarado explícitamente |
| **Identificador** | Verificar unicidad y excluir del análisis |
| **Objetivo** | Se preserva sin transformar en esta fase |

In [10]:
df.head()

,hosts_time_as_host_years,neighbourhood_cleansed,property_type,room_type,accommodates,bathrooms,bedrooms,beds,price_quote_price_per_night,number_of_reviews,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value
0,9,Ñuñoa,Private room in home,Private room,2,1.0,NaN,1.0,45647.0,2,5.00,5.00,5.00,5.00,5.00,5.00,5.00
1,0,Recoleta,Private room in condo,Private room,1,NaN,1.0,1.0,19856.0,8,5.00,5.00,5.00,5.00,5.00,4.88,5.00
2,6,Recoleta,Entire rental unit,Entire home/apt,3,1.0,1.0,2.0,46776.0,126,4.74,4.84,4.85,4.81,4.83,4.81,4.79
3,1,Recoleta,Private room in rental unit,Private room,1,1.0,NaN,1.0,25572.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3,Recoleta,Entire rental unit,Entire home/apt,3,1.0,2.0,2.0,107043.5,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# El diccionario se DECLARA con lo que se sabe del dominio y se COMPLETA con lo
# observado en el archivo: no se afirma nada que no se verifique.
DECLARADO = [
    ("hosts_time_as_host_years",                "ordinal", "numero de años como anfitrion"),
    ("neighbourhood_cleansed",            "nominal",       "Comuna"),
    ("property_type",               "nominal",      "Tipo de propiedad"),
    ("room_type",      "nominal",       "Habitación o casa/depa entero"),
    ("accommodates",     "ordinal",       "Nro de huespedes"),
    ("bathrooms",      "ordinal",       "Nro de baños"),
    ("bedrooms",         "ordinal",       "Nro de habitaciones"),
    ("beds",    "ordinal",       "Nro de camas"),
    ("price_quote_price_per_night",               "objetivo",      "Precio por noche"),
    ("number_of_reviews",    "ordinal",       "Nro de reseñas"),
    ("review_scores_rating",            "continua",      "1 a 5 reseña"),
    ("review_scores_accuracy",            "continua",      "1 a 5 reseña"),
    ("review_scores_cleanliness",            "continua",      "1 a 5 reseña"),
    ("review_scores_checkin",            "continua",      "1 a 5 reseña"),
    ("review_scores_communication",            "continua",      "1 a 5 reseña"),
    ("review_scores_location",            "continua",      "1 a 5 reseña"),
    ("review_scores_value",            "continua",      "1 a 5 reseña"),
]
diccionario = pd.DataFrame(DECLARADO, columns=["variable", "rol", "descripcion"])

# Columnas OBSERVADAS: se leen del archivo, no se escriben a mano
diccionario["dtype"] = [str(df_crudo[v].dtype) for v in diccionario["variable"]]
# nunique() cuenta valores distintos: distingue una binaria de una continua
diccionario["n_unicos"] = [int(df_crudo[v].nunique()) for v in diccionario["variable"]]
# Sobre booleanos, mean() devuelve la PROPORCIÓN de True: por 100 da el porcentaje
diccionario["pct_nulos"] = [round(df_crudo[v].isna().mean() * 100, 1)
                            for v in diccionario["variable"]]

# El diccionario es un entregable, no una tabla de paso: se guarda
diccionario.to_csv(DIR_DOCS / "diccionario_variables.csv", index=False)
diccionario

,variable,rol,descripcion,dtype,n_unicos,pct_nulos
0,hosts_time_as_host_years,ordinal,numero de años como anfitrion,int64,16,0.0
1,neighbourhood_cleansed,nominal,Comuna,str,31,0.0
2,property_type,nominal,Tipo de propiedad,str,65,0.0
3,room_type,nominal,Habitación o casa/depa entero,str,4,0.0
4,accommodates,ordinal,Nro de huespedes,int64,16,0.0
5,bathrooms,ordinal,Nro de baños,float64,27,9.4
6,bedrooms,ordinal,Nro de habitaciones,float64,22,12.6
7,beds,ordinal,Nro de camas,float64,25,5.5
8,price_quote_price_per_night,objetivo,Precio por noche,float64,10856,4.6
9,number_of_reviews,ordinal,Nro de reseñas,int64,449,0.0


### Valores atípicos: medir antes de decidir

La sección siguiente imputa con la mediana «porque hay valores extremos». Esa
afirmación hay que **demostrarla**, no enunciarla. Se usa el criterio del rango
intercuartílico:

$$\text{atípico si} \quad x < Q_1 - 1{,}5 \cdot \text{RIC} \quad \text{o} \quad x > Q_3 + 1{,}5 \cdot \text{RIC}$$

In [12]:
def detectar_atipicos_iqr(serie, factor=1.5):
    """Identifica valores atípicos por el criterio del rango intercuartílico.

    Retorna
    -------
    tuple(np.ndarray, float, float)
        Máscara booleana de atípicos, límite inferior y límite superior.
    """
    # dropna() es obligatorio: un NaN propagaría y devolvería NaN como cuartil
    valores = serie.dropna().to_numpy(dtype="float64")

    # np.percentile con una lista devuelve varios percentiles de una vez
    q1, q3 = np.percentile(valores, [25, 75])
    ric = q3 - q1                                  # rango intercuartílico
    limite_inf, limite_sup = q1 - factor * ric, q3 + factor * ric

    # El operador | es el "o" elemento a elemento de pandas. Con 'or' fallaría:
    # 'or' espera un único booleano, no una serie completa.
    mascara = (serie < limite_inf) | (serie > limite_sup)
    # Comparar con NaN da False pero deja NA: fillna(False) lo hace explícito
    return mascara.fillna(False).to_numpy(), limite_inf, limite_sup


filas = []
for col in ["hosts_time_as_host_years", "accommodates", "bathrooms", "bedrooms", "beds", "number_of_reviews", "price_quote_price_per_night", "review_scores_value", "review_scores_location", "review_scores_communication", "review_scores_checkin", "review_scores_accuracy", "review_scores_cleanliness", "review_scores_rating"]:
    mascara, inf, sup = detectar_atipicos_iqr(df[col])
    filas.append({
        "variable": col,
        "limite_inf": round(inf, 2),
        "limite_sup": round(sup, 2),
        "n_atipicos": int(mascara.sum()),          # sobre booleanos, sum() cuenta True
        "pct_atipicos": round(mascara.mean() * 100, 2),
        "media": round(df[col].mean(), 2),
        "mediana": round(df[col].median(), 2),
    })
pd.DataFrame(filas)

,variable,limite_inf,limite_sup,n_atipicos,pct_atipicos,media,mediana
0,hosts_time_as_host_years,-8.00,16.00,0,0.00,3.91,3.00
1,accommodates,-1.00,7.00,541,2.92,3.11,3.00
2,bathrooms,0.25,2.25,648,3.50,1.31,1.00
3,bedrooms,-0.50,3.50,489,2.64,1.49,1.00
4,beds,-2.00,6.00,446,2.41,2.10,2.00
5,number_of_reviews,-61.00,107.00,1668,9.00,37.23,13.00
6,price_quote_price_per_night,-32421.50,164702.50,1500,8.09,118197.74,59000.00
7,review_scores_value,4.30,5.37,893,4.82,4.76,4.85
8,review_scores_location,4.47,5.32,919,4.96,4.83,4.92
9,review_scores_communication,4.58,5.26,1191,6.43,4.85,4.95


## 3. Limpieza de datos

Limpiar significa dejar los datos completos y consistentes. Tomamos dos decisiones, cada
una justificada técnicamente:

**a) Eliminar `id`.** Es un identificador único (un número distinto por paciente). No
contiene información que ayude a predecir el accidente cerebrovascular, así que lo
descartamos para que no introduzca ruido.

**b) Imputar `bmi` con la mediana.** `bmi` (índice de masa corporal) es la única columna
con valores faltantes. "Imputar" es rellenar esos huecos con un valor representativo.
Tenemos dos opciones:
- La **media** (promedio): se ve arrastrada por los valores extremos.
- La **mediana** (valor central): es **robusta** ante valores atípicos.

Como el `bmi` tiene valores atípicos (se ven en el *boxplot* del final), usar la mediana
evita que esos extremos sesguen el relleno. La función `imputar_nulos_numericos` permite
elegir la estrategia con un parámetro e informa cuántos nulos rellenó y con qué valor
(trazabilidad).

In [13]:
def imputar_nulos_numericos(df, columna, estrategia="mediana"):
    """
    Imputa los valores nulos de una columna numerica.

    Parametros
    ----------
    df : pd.DataFrame
    columna : str
        Columna numerica a imputar.
    estrategia : str
        'media' o 'mediana'.

    Retorna
    -------
    pd.DataFrame
        DataFrame con la columna imputada.
    """
    if columna not in df.columns:
        raise KeyError(f"La columna '{columna}' no existe en el DataFrame.")
    # isnull() da True/False por celda; sum() cuenta los True.
    # Se calcula ANTES de imputar: después ya no habría nulos que contar.
    n_nulos = int(df[columna].isnull().sum())
    if estrategia == "media":
        valor = df[columna].mean()
    elif estrategia == "mediana":
        valor = df[columna].median()
    else:
        raise ValueError("estrategia debe ser 'media' o 'mediana'")
    df = df.copy()
    df[columna] = df[columna].fillna(valor)
    print(f"'{columna}': {n_nulos} nulos imputados con la {estrategia} = {valor:.2f}")
    return df

**Antes** de limpiar, confirmamos dónde están los nulos:

In [14]:
df.isnull().sum()

hosts_time_as_host_years          0
neighbourhood_cleansed            0
property_type                     0
room_type                         0
accommodates                      0
bathrooms                      1747
bedrooms                       2337
beds                           1024
price_quote_price_per_night     847
number_of_reviews                 0
review_scores_rating           3291
review_scores_accuracy         3291
review_scores_cleanliness      3291
review_scores_checkin          3291
review_scores_communication    3291
review_scores_location         3292
review_scores_value            3291
dtype: int64

Eliminamos el identificador `id`:

Imputamos los nulos de `bmi` con la mediana:

In [15]:
is_null = ["bathrooms","bedrooms","beds","price_quote_price_per_night","review_scores_rating","review_scores_accuracy","review_scores_cleanliness","review_scores_checkin","review_scores_communication","review_scores_location","review_scores_value"]

In [16]:
for x in is_null:
    df = imputar_nulos_numericos(df, x, estrategia="mediana")
df

'bathrooms': 1747 nulos imputados con la mediana = 1.00
'bedrooms': 2337 nulos imputados con la mediana = 1.00
'beds': 1024 nulos imputados con la mediana = 2.00
'price_quote_price_per_night': 847 nulos imputados con la mediana = 59000.00
'review_scores_rating': 3291 nulos imputados con la mediana = 4.88
'review_scores_accuracy': 3291 nulos imputados con la mediana = 4.91
'review_scores_cleanliness': 3291 nulos imputados con la mediana = 4.86
'review_scores_checkin': 3291 nulos imputados con la mediana = 4.94
'review_scores_communication': 3291 nulos imputados con la mediana = 4.95
'review_scores_location': 3292 nulos imputados con la mediana = 4.92
'review_scores_value': 3291 nulos imputados con la mediana = 4.85


,hosts_time_as_host_years,neighbourhood_cleansed,property_type,room_type,accommodates,bathrooms,bedrooms,beds,price_quote_price_per_night,number_of_reviews,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value
0,9,Ñuñoa,Private room in home,Private room,2,1.0,1.0,1.0,45647.0,2,5.00,5.00,5.00,5.00,5.00,5.00,5.00
1,0,Recoleta,Private room in condo,Private room,1,1.0,1.0,1.0,19856.0,8,5.00,5.00,5.00,5.00,5.00,4.88,5.00
2,6,Recoleta,Entire rental unit,Entire home/apt,3,1.0,1.0,2.0,46776.0,126,4.74,4.84,4.85,4.81,4.83,4.81,4.79
3,1,Recoleta,Private room in rental unit,Private room,1,1.0,1.0,1.0,25572.0,0,4.88,4.91,4.86,4.94,4.95,4.92,4.85
4,3,Recoleta,Entire rental unit,Entire home/apt,3,1.0,2.0,2.0,107043.5,0,4.88,4.91,4.86,4.94,4.95,4.92,4.85
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18529,6,Las Condes,Entire rental unit,Entire home/apt,4,2.0,2.0,2.0,167533.0,38,4.71,4.71,4.95,4.68,4.82,4.97,4.68
18530,0,Santiago,Entire rental unit,Entire home/apt,6,2.0,2.0,4.0,43492.5,0,4.88,4.91,4.86,4.94,4.95,4.92,4.85
18531,2,Las Condes,Entire rental unit,Entire home/apt,6,1.5,2.0,4.0,137380.0,39,4.79,4.85,4.69,4.90,4.92,4.95,4.74
18532,2,Santiago,Entire rental unit,Entire home/apt,2,1.0,1.0,1.0,55192.5,20,4.70,4.65,4.85,4.85,4.55,4.70,4.65


### Comparar los métodos antes de elegir

La celda anterior aplica la mediana. Falta lo que la rúbrica evalúa: **por qué
esa y no otra**. Ninguna estrategia es mejor en abstracto.

| Método | Cuándo conviene | Qué distorsiona |
| --- | --- | --- |
| **Eliminar filas** | Faltantes escasos y aparentemente aleatorios | Pierde muestra; sesga si no son aleatorios |
| **Media** | Distribución aproximadamente simétrica | La arrastran los extremos; reduce la dispersión |
| **Mediana** | Hay valores extremos o asimetría | Reduce la dispersión; ignora las demás variables |
| **Mediana por grupo** | El valor depende de otra variable observada | Reduce la dispersión dentro de cada grupo |

**Cómo se lee.** `cambio_desv_%` es la columna que decide: toda imputación por
un valor central concentra los datos y **reduce artificialmente la desviación
estándar**. El método que menos la reduce es el que menos deforma la variable.
`filas_perdidas` muestra el costo de eliminar, y `asimetria` indica si la media
representa bien el centro.

> **Si las filas dan casi lo mismo**, la variable es simétrica y las estrategias
> son equivalentes: lo correcto es declararlo y elegir la más simple. Escribir
> «se descarta la media por la asimetría» cuando la tabla muestra asimetría nula
> es exactamente la incoherencia que la rúbrica penaliza.

> **Atención — fuga de datos.** Aquí el valor de relleno se calcula sobre todo
> el conjunto porque no hay partición entre entrenamiento y prueba. Cuando la
> haya, debe calcularse **solo con el conjunto de entrenamiento**: usar la
> mediana global filtra información del conjunto de prueba e infla los
> resultados. El mismo cuidado aplica al escalamiento.

**Después** de limpiar, verificamos que ya no quede ningún nulo (comprobación intermedia):

In [17]:
df.isnull().sum()

hosts_time_as_host_years       0
neighbourhood_cleansed         0
property_type                  0
room_type                      0
accommodates                   0
bathrooms                      0
bedrooms                       0
beds                           0
price_quote_price_per_night    0
number_of_reviews              0
review_scores_rating           0
review_scores_accuracy         0
review_scores_cleanliness      0
review_scores_checkin          0
review_scores_communication    0
review_scores_location         0
review_scores_value            0
dtype: int64

## 4. Transformación: codificación One-Hot

Los modelos solo entienden números, no texto. Por eso hay que convertir las variables de
texto en números. Pero **cómo** las convertimos importa:

- `neighbourhood_cleansed`, `property_type` y `room_type` son

### El patrón `fit` / `transform`

Casi todas las herramientas de scikit-learn funcionan en dos tiempos:
- **`fit`** = *aprender*: la herramienta mira los datos y guarda lo que necesita (por
  ejemplo, qué categorías existen).
- **`transform`** = *aplicar*: usa lo aprendido para convertir los datos.

Dentro de la función `codificar_one_hot` este patrón aparece **dos veces** (una con
`LabelEncoder` y otra con `OneHotEncoder`). Paso a paso, con el ejemplo `["Yes","No","Yes","No"]`:

1. **`le = preprocessing.LabelEncoder()`** — crea la herramienta que convierte texto en
   enteros. Aún está vacía.
2. **`le.fit(datos)`** — aprende las categorías y les asigna un número en orden alfabético:
   `"No" → 0`, `"Yes" → 1`.
3. **`le.transform(datos)`** — aplica lo aprendido → `[1, 0, 1, 0]`. Esto ya es *Label
   Encoding*.
4. **`ohe = preprocessing.OneHotEncoder()`** — crea la segunda herramienta, para One-Hot.
5. **`d = datos_codificados.reshape(-1, 1)`** — reorganiza los datos de una fila a una
   columna, porque el `OneHotEncoder` exige formato de tabla (2 dimensiones). El `-1`
   significa "calcula tú las filas" y el `1` significa "una columna".
6. **`ohe.fit(d)` y `ohe.transform(d).toarray()`** — expande cada entero en columnas
   binarias. `.toarray()` convierte el resultado a un arreglo normal.
7. **`pd.DataFrame(...)` + `pd.concat(...)`** — empaqueta las nuevas columnas con nombres
   legibles y las une al `DataFrame`, eliminando la columna original.

**Mejoras respecto al código original:** la función (a) asigna los nombres legibles
directamente, en lugar de renombrar por número de posición; (b) conserva el índice original
(`index=df.index`) para que la unión no desalinee filas; y (c) **valida** que la cantidad
de nombres coincida con la cantidad de categorías, avisando con un error claro si no.

In [18]:
def codificar_one_hot(df, columna, nombres_columnas):
    """
    Codifica una variable categorica nominal aplicando LabelEncoder y luego
    OneHotEncoder (patron fit/transform de scikit-learn).

    Parametros
    ----------
    df : pd.DataFrame
    columna : str
        Columna categorica a codificar.
    nombres_columnas : list[str]
        Nombres de las nuevas columnas binarias. El orden debe coincidir con el
        orden alfabetico de las categorias (criterio interno de LabelEncoder).

    Retorna
    -------
    pd.DataFrame
        DataFrame con las nuevas columnas one-hot y sin la columna original.

    Lanza
    -----
    KeyError
        Si la columna no existe.
    ValueError
        Si el numero de nombres no coincide con el numero de categorias.
    """
    if columna not in df.columns:
        raise KeyError(f"La columna '{columna}' no existe en el DataFrame.")

    # Paso 1-3: LabelEncoder aprende las categorias y las convierte a enteros
    le = preprocessing.LabelEncoder()
    datos = df[columna]
    le.fit(datos)
    datos_codificados = le.transform(datos)

    # Verificacion: los nombres deben corresponder a las categorias detectadas
    if len(nombres_columnas) != len(le.classes_):
        raise ValueError(
            f"Se esperaban {len(le.classes_)} nombres para '{columna}' "
            f"({list(le.classes_)}), pero se recibieron {len(nombres_columnas)}."
        )

    # Paso 4-6: OneHotEncoder expande los enteros a columnas binarias
    ohe = preprocessing.OneHotEncoder()
    d = datos_codificados.reshape(-1, 1)   # formato 2D requerido por el encoder
    ohe.fit(d)
    matriz = ohe.transform(d).toarray()

    # Paso 7: empaquetamos con nombres legibles y conservamos el indice original
    nuevas = pd.DataFrame(matriz, columns=nombres_columnas, index=df.index).astype(int)

    df = df.drop(columns=[columna]).reset_index(drop=True)
    nuevas = nuevas.reset_index(drop=True)
    return pd.concat([df, nuevas], axis=1)

### a. `neighbourhood_cleansed`
Categorías comunas

In [19]:
# Los nombres van en el ORDEN ALFABÉTICO de las categorías, que es el criterio
# interno de LabelEncoder. Si se invierten, las columnas
# quedan mal rotuladas y ningún error lo advierte.
lista_comunas = df["neighbourhood_cleansed"].unique().tolist()
lista_comunas.sort()
index = 0
for x in lista_comunas:
    lista_comunas[index] = "comuna_" + x
    index +=1
print(lista_comunas)

['comuna_Cerrillos', 'comuna_Cerro Navia', 'comuna_Conchalí', 'comuna_El Bosque', 'comuna_Estación Central', 'comuna_Huechuraba', 'comuna_Independencia', 'comuna_La Cisterna', 'comuna_La Florida', 'comuna_La Granja', 'comuna_La Pintana', 'comuna_La Reina', 'comuna_Las Condes', 'comuna_Lo Barnechea', 'comuna_Lo Espejo', 'comuna_Lo Prado', 'comuna_Macul', 'comuna_Maipú', 'comuna_Pedro Aguirre Cerda', 'comuna_Peñalolén', 'comuna_Providencia', 'comuna_Pudahuel', 'comuna_Quilicura', 'comuna_Quinta Normal', 'comuna_Recoleta', 'comuna_Renca', 'comuna_San Joaquín', 'comuna_San Miguel', 'comuna_Santiago', 'comuna_Vitacura', 'comuna_Ñuñoa']


In [20]:
df = codificar_one_hot(df, "neighbourhood_cleansed", lista_comunas)
df.head()

,hosts_time_as_host_years,property_type,room_type,accommodates,bathrooms,bedrooms,beds,price_quote_price_per_night,number_of_reviews,review_scores_rating,...,comuna_Pudahuel,comuna_Quilicura,comuna_Quinta Normal,comuna_Recoleta,comuna_Renca,comuna_San Joaquín,comuna_San Miguel,comuna_Santiago,comuna_Vitacura,comuna_Ñuñoa
0,9,Private room in home,Private room,2,1.0,1.0,1.0,45647.0,2,5.00,...,0,0,0,0,0,0,0,0,0,1
1,0,Private room in condo,Private room,1,1.0,1.0,1.0,19856.0,8,5.00,...,0,0,0,1,0,0,0,0,0,0
2,6,Entire rental unit,Entire home/apt,3,1.0,1.0,2.0,46776.0,126,4.74,...,0,0,0,1,0,0,0,0,0,0
3,1,Private room in rental unit,Private room,1,1.0,1.0,1.0,25572.0,0,4.88,...,0,0,0,1,0,0,0,0,0,0
4,3,Entire rental unit,Entire home/apt,3,1.0,2.0,2.0,107043.5,0,4.88,...,0,0,0,1,0,0,0,0,0,0


### b. `property_type`


In [21]:
lista_property_type = df["property_type"].unique().tolist()
lista_property_type.sort()
index = 0
for x in lista_property_type:
    loop_aux = x.replace(" ", "_")
    lista_property_type[index] = "tipo_" + loop_aux
    index +=1
print(lista_property_type)

['tipo_Bus', 'tipo_Camper/RV', 'tipo_Casa_particular', 'tipo_Castle', 'tipo_Cave', 'tipo_Entire_bungalow', 'tipo_Entire_cabin', 'tipo_Entire_chalet', 'tipo_Entire_condo', 'tipo_Entire_cottage', 'tipo_Entire_guest_suite', 'tipo_Entire_guesthouse', 'tipo_Entire_home', 'tipo_Entire_loft', 'tipo_Entire_place', 'tipo_Entire_rental_unit', 'tipo_Entire_serviced_apartment', 'tipo_Entire_townhouse', 'tipo_Entire_vacation_home', 'tipo_Entire_villa', 'tipo_Hut', 'tipo_Private_room', 'tipo_Private_room_in_bed_and_breakfast', 'tipo_Private_room_in_bungalow', 'tipo_Private_room_in_bus', 'tipo_Private_room_in_cabin', 'tipo_Private_room_in_casa_particular', 'tipo_Private_room_in_chalet', 'tipo_Private_room_in_condo', 'tipo_Private_room_in_cottage', 'tipo_Private_room_in_dome', 'tipo_Private_room_in_earthen_home', 'tipo_Private_room_in_farm_stay', 'tipo_Private_room_in_guest_suite', 'tipo_Private_room_in_guesthouse', 'tipo_Private_room_in_home', 'tipo_Private_room_in_hostel', 'tipo_Private_room_in_loft

In [22]:
df = codificar_one_hot(df, "property_type", lista_property_type)
df.head()

,hosts_time_as_host_years,room_type,accommodates,bathrooms,bedrooms,beds,price_quote_price_per_night,number_of_reviews,review_scores_rating,review_scores_accuracy,...,tipo_Shared_room_in_casa_particular,tipo_Shared_room_in_guesthouse,tipo_Shared_room_in_home,tipo_Shared_room_in_hostel,tipo_Shared_room_in_hotel,tipo_Shared_room_in_rental_unit,tipo_Tiny_home,tipo_Tower,tipo_Windmill,tipo_Yurt
0,9,Private room,2,1.0,1.0,1.0,45647.0,2,5.00,5.00,...,0,0,0,0,0,0,0,0,0,0
1,0,Private room,1,1.0,1.0,1.0,19856.0,8,5.00,5.00,...,0,0,0,0,0,0,0,0,0,0
2,6,Entire home/apt,3,1.0,1.0,2.0,46776.0,126,4.74,4.84,...,0,0,0,0,0,0,0,0,0,0
3,1,Private room,1,1.0,1.0,1.0,25572.0,0,4.88,4.91,...,0,0,0,0,0,0,0,0,0,0
4,3,Entire home/apt,3,1.0,2.0,2.0,107043.5,0,4.88,4.91,...,0,0,0,0,0,0,0,0,0,0


### c. `room_type`


In [23]:
lista_room_type = df["room_type"].unique().tolist()
lista_room_type.sort()
index = 0
for x in lista_room_type:
    loop_aux = x.replace(" ", "_")
    lista_room_type[index] = "habitacion_" + loop_aux
    index +=1
print(lista_room_type)

['habitacion_Entire_home/apt', 'habitacion_Hotel_room', 'habitacion_Private_room', 'habitacion_Shared_room']


In [24]:
df = codificar_one_hot(df, "room_type", lista_room_type)
df.head()

,hosts_time_as_host_years,accommodates,bathrooms,bedrooms,beds,price_quote_price_per_night,number_of_reviews,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,...,tipo_Shared_room_in_hotel,tipo_Shared_room_in_rental_unit,tipo_Tiny_home,tipo_Tower,tipo_Windmill,tipo_Yurt,habitacion_Entire_home/apt,habitacion_Hotel_room,habitacion_Private_room,habitacion_Shared_room
0,9,2,1.0,1.0,1.0,45647.0,2,5.00,5.00,5.00,...,0,0,0,0,0,0,0,0,1,0
1,0,1,1.0,1.0,1.0,19856.0,8,5.00,5.00,5.00,...,0,0,0,0,0,0,0,0,1,0
2,6,3,1.0,1.0,2.0,46776.0,126,4.74,4.84,4.85,...,0,0,0,0,0,0,1,0,0,0
3,1,1,1.0,1.0,1.0,25572.0,0,4.88,4.91,4.86,...,0,0,0,0,0,0,0,0,1,0
4,3,3,1.0,2.0,2.0,107043.5,0,4.88,4.91,4.86,...,0,0,0,0,0,0,1,0,0,0


## 5. Escalamiento (estandarización)

**Qué es.** Estandarizar (*z-score*) transforma una variable para que tenga **media 0 y
desviación estándar 1**, mediante la fórmula:

$$ z = \frac{x - \mu}{\sigma} $$

donde $\mu$ es la media y $\sigma$ la desviación estándar de la columna.

**Por qué.** Variables como `age` (1–90) y `avg_glucose_level` (55–270) están en escalas muy
distintas. Algunos modelos (KNN, SVM, regresión regularizada) se basan en distancias y darían
más peso a la variable de números más grandes solo por su escala. Estandarizar las pone en
condiciones comparables.

**Qué NO escalamos.** Solo estandarizamos las variables **continuas** (`age`,
`avg_glucose_level`, `bmi`). No tocamos las columnas binarias (las one-hot, `hypertension`,
`heart_disease`) ni la variable objetivo `stroke`, porque estandarizar valores 0/1 o el
*target* distorsiona su interpretación. Esta es una corrección importante frente a aplicar
el escalador a toda la tabla.

La función devuelve también el `scaler` ajustado, por si más adelante hay que aplicar la
misma transformación a datos nuevos.

In [25]:
from sklearn.preprocessing import RobustScaler
def escalar_caracteristicas(df, columnas):
    """
    Estandariza (z-score) las columnas indicadas con StandardScaler.

    Retorna el DataFrame transformado y el objeto scaler ajustado (para poder
    reutilizarlo sobre datos nuevos sin recalcular las estadisticas).
    """
    # copy() evita modificar el DataFrame recibido: quien llama decide si
    # reemplaza el original o conserva ambos.
    df = df.copy()
    scaler = RobustScaler()
    # fit_transform hace dos cosas: aprende media y desviación (fit) y aplica
    # la transformación (transform). Se devuelve el scaler para poder aplicar
    # EXACTAMENTE la misma transformación a datos nuevos con .transform().
    df[columnas] = scaler.fit_transform(df[columnas])
    return df, scaler

Tras escalar, la media debe quedar ≈ 0 y la desviación estándar ≈ 1:

In [26]:
# Solo se escalan las continuas: las binarias y las codificadas ya están en una
# escala comparable, y escalarlas destruiría su interpretación como 0/1.
columnas_continuas = ["hosts_time_as_host_years", "accommodates", "bathrooms", "bedrooms", "beds", "number_of_reviews", "review_scores_value", "review_scores_location", "review_scores_communication", "review_scores_checkin", "review_scores_accuracy", "review_scores_cleanliness", "review_scores_rating"]
df, scaler = escalar_caracteristicas(df, columnas_continuas)
df[columnas_continuas].describe()

,hosts_time_as_host_years,accommodates,bathrooms,bedrooms,beds,number_of_reviews,review_scores_value,review_scores_location,review_scores_communication,review_scores_checkin,review_scores_accuracy,review_scores_cleanliness,review_scores_rating
count,18534.000000,18534.000000,18534.000000,18534.000000,18534.000000,18534.000000,18534.000000,18534.000000,18534.000000,18534.000000,18534.000000,18534.000000,18534.000000
mean,0.151496,0.055223,0.278245,0.431046,0.092587,0.577022,-0.396597,-0.439475,-0.628869,-0.553874,-0.490689,-0.398842,-0.412221
std,0.599525,0.908607,0.712242,0.973766,1.803896,1.585537,1.863933,1.735537,2.385961,2.519610,2.095631,1.624286,1.809040
min,-0.500000,-1.000000,-0.500000,-1.000000,-1.000000,-0.309524,-20.263158,-23.058824,-30.384615,-32.833333,-24.437500,-17.545455,-20.421053
25%,-0.333333,-0.500000,0.000000,0.000000,-1.000000,-0.261905,-0.526316,-0.529412,-0.615385,-0.500000,-0.562500,-0.590909,-0.526316
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.666667,0.500000,0.000000,1.000000,0.000000,0.738095,0.473684,0.470588,0.384615,0.500000,0.437500,0.409091,0.473684
max,2.000000,6.500000,21.000000,49.000000,55.000000,32.738095,0.789474,0.470588,0.384615,0.500000,0.562500,0.636364,0.631579


## 6. Validación técnica y verificación del código

Verificar significa demostrar, con evidencia, que el resultado es correcto. La función
`validar_dataset` ejecuta varias comprobaciones y usa `assert`: si una condición no se
cumple, el notebook se detiene con un error, dejando constancia de la falla.

Comprobamos:
1. **Integridad** — que no queden valores nulos.
2. **Consistencia** — que todas las columnas sean numéricas (no quedó texto sin codificar).
3. **Coherencia** — que cada grupo one-hot sume exactamente 1 por fila: cada paciente
   pertenece a **una sola** categoría dentro de cada variable.
4. **Duplicados** — cuántas filas repetidas hay.
5. **Dimensiones** — la forma final de la tabla.

In [27]:
def validar_dataset(df, grupos_one_hot=None):
    """
    Comprueba integridad, consistencia y coherencia del dataset final.
    Lanza AssertionError si alguna comprobacion falla.
    """
    print("VALIDACION DEL DATASET FINAL")
    print("-" * 45)

    # 1. Integridad: sin valores nulos
    nulos = int(df.isnull().sum().sum())
    assert nulos == 0, f"Quedan {nulos} valores nulos."
    print(f"[OK] Sin valores nulos (total = {nulos})")

    # 2. Consistencia: todas las columnas son numericas
    no_numericas = df.select_dtypes(exclude=[np.number]).columns.tolist()
    assert not no_numericas, f"Columnas no numericas: {no_numericas}"
    print("[OK] Todas las columnas son numericas")

    # 3. Coherencia: cada grupo one-hot suma 1 por fila (una sola categoria activa)
    if grupos_one_hot:
        for nombre, cols in grupos_one_hot.items():
            suma = df[cols].sum(axis=1)
            assert (suma == 1).all(), f"El grupo '{nombre}' no suma 1 en todas las filas."
            print(f"[OK] Grupo one-hot '{nombre}' coherente")

    # 4. Duplicados
    dup = int(df.duplicated().sum())
    print(f"[INFO] Filas duplicadas: {dup}")

    # 5. Dimensiones finales
    print(f"[INFO] Dimensiones finales: {df.shape}")
    return True




## Conclusiones y trazabilidad

El *pipeline* dejó el *dataset* sin valores nulos, con todas las variables en formato
numérico y las continuas estandarizadas. Cada paso se implementó como una función
documentada y reutilizable (`cargar_datos`, `explorar_dataframe`, `imputar_nulos_numericos`,
`codificar_one_hot`, `escalar_caracteristicas`, `validar_dataset`), garantizando un flujo
modular, reproducible y verificable.

**Trazabilidad con el repositorio (F2):** este notebook se ubica en la carpeta `F2/` del
repositorio GitHub; el `README` documenta las dependencias (`numpy`, `pandas`,
`scikit-learn`, `matplotlib`) y las instrucciones de ejecución. El historial de *commits*
refleja el avance del *pipeline* descrito en el informe técnico de la Fase 2.

---

## 7. Recursividad: aplanar los metadatos del proyecto

Todo el procesamiento anterior fue **estructurado**: secuencias, condicionales y
bucles. Esta sección introduce la **recursividad**, que es la herramienta natural
cuando la estructura tiene profundidad desconocida. El indicador ID 2.1 la
contempla explícitamente.

El caso es real y no decorativo: los metadatos del proyecto forman un diccionario
anidado y, para exportarlos como tabla, hay que convertirlos en pares
clave-valor. Un bucle no sirve porque no se sabe cuántos niveles hay.

**Anatomía de la función recursiva:**
- *Caso base*: el valor no es un diccionario → se devuelve el par clave-valor.
- *Caso recursivo*: el valor es un diccionario → la función se llama a sí misma
  un nivel más abajo, arrastrando el prefijo de la ruta.

In [28]:
# Los metadatos del proyecto, con la estructura anidada que corresponde
METADATOS = {
    "proyecto": {
        "nombre": "datos airbnb",
        "fase": "F2",
        "semilla": 42,
    },
    "datos": {
        "archivo": "listings.csv.gz",
        "fuente": {
            "plataforma": "InsideAirbnb",
            "autor": "InsideAirbnb",
            "url": "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/data/reviews.csv.gz",
        },
    },
    "decisiones": {
        "imputacion": {"datos_continuos": "mediana"},
        "escalador": "Robust",
        "categoricas": ["neighbourhood_cleansed", "property_type", "room type"],
    },
}


def aplanar(estructura, prefijo="", separador="."):
    """Aplana un diccionario anidado en uno de un solo nivel.

    Ejemplo
    -------
    >>> aplanar({"a": {"b": 1, "c": {"d": 2}}})
    {'a.b': 1, 'a.c.d': 2}
    """
    plano = {}
    for clave, valor in estructura.items():
        # Se construye la ruta completa: en el primer nivel no hay prefijo
        ruta = f"{prefijo}{separador}{clave}" if prefijo else str(clave)

        if isinstance(valor, dict):
            # CASO RECURSIVO: la función se llama a sí misma un nivel más abajo
            plano.update(aplanar(valor, ruta, separador))
        elif isinstance(valor, (list, tuple)):
            # Las secuencias se convierten en texto legible para la tabla
            plano[ruta] = ", ".join(str(v) for v in valor)
        else:
            # CASO BASE: el valor es simple, se devuelve el par
            plano[ruta] = valor
    return plano


metadatos_planos = aplanar(METADATOS)
print(f"El diccionario anidado se aplanó en {len(metadatos_planos)} pares.\n")
for clave, valor in metadatos_planos.items():
    print(f"  {clave:<28} {str(valor)[:52]}")

El diccionario anidado se aplanó en 10 pares.

  proyecto.nombre              datos airbnb
  proyecto.fase                F2
  proyecto.semilla             42
  datos.archivo                listings.csv.gz
  datos.fuente.plataforma      InsideAirbnb
  datos.fuente.autor           InsideAirbnb
  datos.fuente.url             https://data.insideairbnb.com/chile/rm/santiago/2026
  decisiones.imputacion.datos_continuos mediana
  decisiones.escalador         Robust
  decisiones.categoricas       neighbourhood_cleansed, property_type, room type


In [29]:
# Verificación de la función recursiva: casos normal, profundo y límite
assert aplanar({"a": 1}) == {"a": 1}, "Caso plano"
assert aplanar({"a": {"b": 1}}) == {"a.b": 1}, "Un nivel de anidamiento"
assert aplanar({"a": {"b": {"c": {"d": 4}}}}) == {"a.b.c.d": 4}, "Anidamiento profundo"
assert aplanar({}) == {}, "Caso límite: diccionario vacío"
assert aplanar({"x": [1, 2, 3]}) == {"x": "1, 2, 3"}, "Listas convertidas a texto"
print("[OK] La función recursiva pasa las cinco pruebas.")

[OK] La función recursiva pasa las cinco pruebas.


---

## 8. Persistencia y trazabilidad

El resultado debe quedar guardado y documentado. Esta sección produce los
artefactos que se versionan en el repositorio y se citan en el informe.

In [30]:
ARCHIVO_PROCESADO = DIR_PROCESADO / "airbnb_procesado.csv"

# index=False evita agregar una columna con el número de fila, que no es un dato
df.to_csv(ARCHIVO_PROCESADO, index=False)

# Verificación de ida y vuelta: se relee lo guardado y se comprueba que coincide.
# Si el archivo se escribió mal, es preferible descubrirlo ahora.
releido = pd.read_csv(ARCHIVO_PROCESADO)
assert releido.shape == df.shape, "El archivo releído no coincide en dimensiones"
print(f"Guardado y verificado: {ARCHIVO_PROCESADO}")
print(f"Tamaño en disco: {ARCHIVO_PROCESADO.stat().st_size / 1024:.1f} kB")

# Los metadatos aplanados con la función recursiva de la sección 7
pd.DataFrame(list(metadatos_planos.items()), columns=["clave", "valor"]).to_csv(
    DIR_DOCS / "metadatos_proyecto.csv", index=False)
print(f"Metadatos exportados a {DIR_DOCS}/metadatos_proyecto.csv")

Guardado y verificado: Data\Processed\airbnb_procesado.csv
Tamaño en disco: 6743.7 kB
Metadatos exportados a docs/metadatos_proyecto.csv


In [31]:
# Resumen comparativo entre el punto de partida y el resultado.
# Esta tabla responde de una vez a la pregunta "qué cambió".
resumen = pd.DataFrame([
    {"indicador": "Filas", "inicial": df_crudo.shape[0], "final": df.shape[0]},
    {"indicador": "Columnas", "inicial": df_crudo.shape[1], "final": df.shape[1]},
    {"indicador": "Valores nulos",
     "inicial": int(df_crudo.isna().sum().sum()), "final": int(df.isna().sum().sum())},
    {"indicador": "Columnas de texto",
     "inicial": int((df_crudo.dtypes == "object").sum()),
     "final": int((df.dtypes == "object").sum())},
    {"indicador": "Filas duplicadas",
     "inicial": int(df_crudo.duplicated().sum()), "final": int(df.duplicated().sum())},
])
resumen

,indicador,inicial,final
0,Filas,18534,18534
1,Columnas,90,114
2,Valores nulos,318357,0
3,Columnas de texto,0,0
4,Filas duplicadas,0,167
